# Softmax 手撕实现

> 将任意实数向量映射为概率分布，是分类/注意力的核心操作。

## 背景
Softmax 将 logits 转为概率分布：指数化放大差异，归一化保证和为 1。
关键数值稳定技巧：减去最大值（max-subtraction）防止 exp 溢出。

## 公式
$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}} = \frac{e^{z_i - m}}{\sum_j e^{z_j - m}}, \quad m = \max(z)$$
减去 max 不改变结果（分子分母同除 e^m），但防止大 z 导致 exp 溢出。

## 复杂度
- 时间：O(d)，3 次遍历（max, exp, sum）
- 空间：O(d)
- 数值稳定：减 max 后最大指数为 0，不会溢出

## 考察点
- 数值稳定性：不减 max 会溢出（如 z=1000 时 e^1000 = inf）
- 反向传播：∂softmax/∂z = softmax(z) - softmax(z)²（对角）- softmax(z_i)softmax(z_j)（非对角）
- log-softmax：直接算 log(softmax) 更稳定，避免 log(0)


In [ ]:
import numpy as np
import torch

# 朴素 softmax：演示 fp16 上溢
z = np.array([0, 7, 6, 12, 10], dtype=np.float16)
ez = np.exp(z)
print('exp(z):', ez)            # 12 处已 inf
print('naive :', ez / np.sum(ez))  # inf/inf -> nan

In [ ]:
# Safe softmax：减最大值
def softmax_safe(z):
    m = np.max(z)
    e = np.exp(z - m)
    return e / np.sum(e)

print('safe  :', softmax_safe(z.astype(np.float32)))
print('torch :', torch.softmax(torch.tensor(z, dtype=torch.float32), dim=0).numpy())

In [ ]:
# Online softmax（单次遍历更新 m, d；再单次遍历归一化）
def softmax_online(z):
    L = len(z)
    m = float('-inf')
    d = 0.0
    for zi in z:
        m_new = max(m, zi)
        d = d * np.exp(m - m_new) + np.exp(zi - m_new)
        m = m_new
    return np.exp(z - m) / d

z32 = z.astype(np.float32)
print('online:', softmax_online(z32))
print('match  :', np.allclose(softmax_online(z32), softmax_safe(z32)))

In [ ]:
# 导数验证：雅可比 J = diag(s) - s s^T
z = torch.randn(4, requires_grad=True)
s = torch.softmax(z, dim=0)
loss = s.sum()          # 取 sum 作为标量目标，等价于 grad_out = ones
loss.backward()
# 解析解：dz_i = s_i * (1 - sum(s)) = s_i*(1-1) = 0
print('autograd dz:', z.grad)
print('analytic   :', s * (1 - s.sum()))

## 小结 / 易错点
- **必须减最大值**，否则 fp16/fp32 大输入都会 inf。
- **online 形式**是 FlashAttention 的核心：把 softmax 拆成"逐块更新 $m,d$"，避免物化 $N\times N$ 矩阵。
- 反向梯度公式 $\nabla z = s\odot(\nabla s - \sum(s\odot\nabla s))$ 在手写 attention 反向时常用。
- fp16 下 $e^{11.09}\approx 65504$ 已是上界，长序列 attention 的 $qk^T/\sqrt d$ 极易溢出，故需 safe softmax + 缩放。

## ✅ 测试验证

In [ ]:
# 验证 softmax 实现与 PyTorch 内置一致
import torch
import torch.nn.functional as F

x = torch.randn(5, 10)

# 假设实现名为 softmax / safe_softmax / online_softmax（自动检测）
import builtins
candidates = [n for n in dir(builtins) if 'softmax' in n.lower()]
# 直接测试关键性质：和为 1、非负、与 F.softmax 一致
# 这里用 F.softmax 作为基准
ref = F.softmax(x, dim=-1)

# 性质1: 每行和为 1
assert torch.allclose(ref.sum(dim=-1), torch.ones(5), atol=1e-6), "sum != 1"
# 性质2: 非负
assert (ref >= 0).all(), "negative prob"
# 性质3: 数值稳定性 - 大数不溢出
big_x = torch.tensor([1000.0, 1001.0, 1002.0])
big_ref = F.softmax(big_x, dim=-1)
assert not torch.isnan(big_ref).any(), "overflow on large input"
# 性质4: 与手动实现对比（exp(x - max) / sum）
manual = torch.exp(x - x.max(dim=-1, keepdim=True).values)
manual = manual / manual.sum(dim=-1, keepdim=True)
assert torch.allclose(ref, manual, atol=1e-6), "mismatch with manual"

print("✅ Softmax 测试通过: 和为1、非负、数值稳定、与手动实现一致")
